# Linear Regression

## Table of Contents
1. [Introduction](#1-introduction)
2. [Simple Linear Regression](#2-simple-linear-regression)
3. [Multiple Linear Regression](#3-multiple-linear-regression)
4. [Gradient Descent from Scratch](#4-gradient-descent-from-scratch)
5. [Regularization (Ridge, Lasso, Elastic Net)](#5-regularization)
6. [Polynomial Regression](#6-polynomial-regression)
7. [Model Evaluation & Diagnostics](#7-model-evaluation)
8. [Real-World Example: Diabetes Dataset](#8-diabetes-example)

---

## 1. Introduction

Linear regression is a **supervised learning** algorithm for predicting continuous values. It finds the best-fitting line that minimizes the difference between predicted and actual values.

**Key Formula:**
```
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ
```

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.datasets import make_regression, load_diabetes

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

## 2. Simple Linear Regression

**Equation:** `y = mx + b`

One feature predicting one target variable.

In [ ]:
# Generate sample data
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Results
print(f"Coefficient (slope): {model.coef_[0][0]:.4f}")
print(f"Intercept: {model.intercept_[0]:.4f}")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', alpha=0.6, label='Actual')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simple Linear Regression')
plt.legend()
plt.show()

---

## 3. Multiple Linear Regression

**Equation:** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ`

Multiple features predicting one target.

In [ ]:
# Generate data with multiple features
X, y = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)

feature_names = ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"\nCoefficients:")
for name, coef in zip(feature_names, model.coef_):
    print(f"  {name}: {coef:.4f}")

In [ ]:
# Feature importance visualization
plt.figure(figsize=(10, 5))
plt.barh(feature_names, model.coef_)
plt.xlabel('Coefficient Value')
plt.title('Feature Coefficients')
plt.show()

---

## 4. Gradient Descent from Scratch

The optimization algorithm that minimizes the cost function:
```
θⱼ := θⱼ - α * ∂J(θ)/∂θⱼ
```

In [ ]:
# Gradient Descent Implementation
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

m = len(X)
X_b = np.c_[np.ones((m, 1)), X]  # Add x0 = 1 for intercept

learning_rate = 0.1
n_iterations = 1000
theta = np.random.randn(2, 1)

for iteration in range(n_iterations):
    gradients = (2/m) * X_b.T.dot(X_b.dot(theta) - y)
    theta = theta - learning_rate * gradients

print(f"Intercept (θ₀): {theta[0][0]:.4f}")
print(f"Slope (θ₁): {theta[1][0]:.4f}")

---

## 5. Regularization

### Ridge (L2): Adds λ * Σθⱼ² - shrinks coefficients

### Lasso (L1): Adds λ * Σ|θⱼ| - can set coefficients to zero

### Elastic Net: Combines L1 + L2

In [ ]:
# Compare Regularization Methods
X, y = make_regression(n_samples=200, n_features=10, noise=20, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

# Lasso
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

# Elastic Net
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)

print("R² Scores:")
print(f"  Ridge: {r2_score(y_test, ridge.predict(X_test_scaled)):.4f}")
print(f"  Lasso: {r2_score(y_test, lasso.predict(X_test_scaled)):.4f}")
print(f"  Elastic Net: {r2_score(y_test, elastic.predict(X_test_scaled)):.4f}")

print(f"\nNon-zero coefficients:")
print(f"  Ridge: {np.sum(ridge.coef_ != 0)}")
print(f"  Lasso: {np.sum(lasso.coef_ != 0)}")
print(f"  Elastic Net: {np.sum(elastic.coef_ != 0)}")

---

## 6. Polynomial Regression

For non-linear relationships by adding polynomial features.

In [ ]:
# Generate non-linear data
X = 6 * np.random.rand(100, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.randn(100, 1)

# Transform to polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

# Train
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"R² Score: {r2_score(y_test, model.predict(X_test)):.4f}")

In [ ]:
# Visualize polynomial fit
X_range = np.linspace(-3, 3, 100).reshape(-1, 1)
X_range_poly = poly.transform(X_range)
y_range = model.predict(X_range_poly)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='blue', alpha=0.6, label='Data')
plt.plot(X_range, y_range, color='red', linewidth=2, label='Polynomial Fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial Regression (Degree 2)')
plt.legend()
plt.show()

---

## 7. Model Evaluation & Diagnostics

### Key Metrics:
- **MSE**: Mean Squared Error
- **RMSE**: Root Mean Squared Error
- **MAE**: Mean Absolute Error
- **R²**: Coefficient of Determination

### Assumptions Check:
- Linearity
- Homoscedasticity
- Normality of residuals

In [ ]:
# Generate data and train model
X, y = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
residuals = y_test - y_pred.flatten()

# Calculate all metrics
print("Evaluation Metrics:")
print(f"  MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"  R² Score: {r2_score(y_test, y_pred):.4f}")

In [ ]:
# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Fitted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Q-Q Plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Histogram of residuals
axes[1, 0].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Residuals Distribution')

# Actual vs Predicted
axes[1, 1].scatter(y_test, y_pred, alpha=0.6)
axes[1, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1, 1].set_xlabel('Actual')
axes[1, 1].set_ylabel('Predicted')
axes[1, 1].set_title('Actual vs Predicted')

plt.tight_layout()
plt.show()

---

## 8. Diabetes Dataset Example

Real-world example predicting disease progression.

In [ ]:
# Load diabetes dataset
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

In [ ]:
# Feature importance
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient')
plt.title('Diabetes: Feature Importance')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))

---

## Summary

| Method | Use Case | Pros | Cons |
|--------|----------|------|------|
| Simple Linear | One feature | Interpretable | Limited |
| Multiple Linear | Multiple features | More predictive | Multicollinearity |
| Polynomial | Non-linear | Flexible | Overfitting |
| Ridge (L2) | Many features | Prevents overfitting | Doesn't select features |
| Lasso (L1) | Feature selection | Sparse models | Unstable |
| Elastic Net | Best of both | Balanced | More parameters |